# 05 — Explainability & Fairness
1. **Why** does the model flag a patient? (SHAP values)
2. Does it work **equally well** across age, sex and race?

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

PROC, FIG, RES, MOD = Path("data/processed"), Path("figures"), Path("results"), Path("models")

df = pd.read_parquet(PROC / "model_table.parquet")
test_patients = set(pd.read_parquet(PROC / "test_patients.parquet")["patient_nbr"])
test = df[df["patient_nbr"].isin(test_patients)].reset_index(drop=True)
X_test, y_test = test.drop(columns=["readmit_30", "patient_nbr"]), test["readmit_30"]

calibrated = joblib.load(MOD / "lgbm_calibrated.joblib")
lgbm = calibrated.estimator.estimator          # the tuned LightGBM inside the calibration wrapper
p_test = calibrated.predict_proba(X_test)[:, 1]
print(f"Test: {len(X_test):,}  ROC-AUC {roc_auc_score(y_test, p_test):.3f}")

## 1. SHAP values
LightGBM computes exact SHAP values natively (`pred_contrib=True`). Each value is a feature's push on the log-odds of readmission for one patient.

In [ ]:
contrib = lgbm.predict(X_test, pred_contrib=True)
shap_vals, base_value = contrib[:, :-1], contrib[0, -1]
shap_df = pd.DataFrame(shap_vals, columns=X_test.columns)

importance = shap_df.abs().mean().sort_values(ascending=False)
top15 = importance.head(15)[::-1]

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top15.index, top15.values, color="#2a6f97")
ax.set_xlabel("Mean |SHAP value| (log-odds)")
ax.set_title("Top 15 drivers of predicted readmission risk")
plt.tight_layout()
plt.savefig(FIG / "05_shap_importance.png", dpi=150)
plt.show()
importance.head(15).round(4)

## 2. How the top numeric drivers shape risk

In [ ]:
num_top = [c for c in importance.index if pd.api.types.is_numeric_dtype(X_test[c])][:4]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.8), sharey=True)
for ax, col in zip(axes, num_top):
    ax.scatter(X_test[col], shap_df[col], s=4, alpha=0.3, color="#2a6f97")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_xlabel(col)
axes[0].set_ylabel("SHAP value (log-odds)")
fig.suptitle("SHAP dependence: higher = pushes risk up")
plt.tight_layout()
plt.savefig(FIG / "05_shap_dependence.png", dpi=150)
plt.show()

## 3. One patient, explained

In [ ]:
i = int(np.argmax(p_test))   # the highest-risk patient in the test set
print(f"Predicted 30-day readmission risk: {p_test[i]:.1%}   Actually readmitted: {bool(y_test.iloc[i])}")

pt = pd.DataFrame({"feature": X_test.columns, "value": X_test.iloc[i].astype(str).values,
                   "shap": shap_vals[i]})
pt = pt.reindex(pt["shap"].abs().sort_values(ascending=False).index).head(12)[::-1]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh([f"{f} = {v}" for f, v in zip(pt["feature"], pt["value"])], pt["shap"],
        color=["#c1666b" if s > 0 else "#2a6f97" for s in pt["shap"]])
ax.axvline(0, color="grey", lw=0.8)
ax.set_xlabel("SHAP value (log-odds): red raises risk, blue lowers it")
ax.set_title(f"Why this patient was flagged (predicted risk {p_test[i]:.0%})")
plt.tight_layout()
plt.savefig(FIG / "05_patient_explained.png", dpi=150)
plt.show()

## 4. Fairness across groups
For each group: can the model still rank risk (**ROC-AUC**)? Is it **calibrated** (predicted vs actual rate)? And who ends up on the **top-10% follow-up list**?

In [ ]:
# rank-based: exactly the top 10% of patients, no matter how many share a score
k = int(0.10 * len(p_test))
flagged = np.zeros(len(p_test), dtype=bool)
flagged[np.argsort(p_test)[::-1][:k]] = True
print(f"Flagged: {flagged.sum():,} patients ({flagged.mean():.1%})")

groups = {
    "Sex":  np.where(X_test["male"] == 1, "Male", "Female"),
    "Age":  pd.cut(X_test["age_num"], [0, 40, 60, 80, 100], labels=["<40", "40-59", "60-79", "80+"]).astype(str),
    "Race": X_test["race"].astype(str).values,
}

rows = []
for attr, g in groups.items():
    for level in pd.unique(g):
        m = g == level
        if m.sum() < 200:          # too small to judge reliably
            continue
        yt, pt, fl = y_test[m], p_test[m], flagged[m]
        rows.append({"attribute": attr, "group": level, "n": int(m.sum()),
                     "roc_auc": round(roc_auc_score(yt, pt), 3),
                     "actual_rate": round(yt.mean(), 3),
                     "predicted_rate": round(pt.mean(), 3),
                     "flagged_share": round(fl.mean(), 3),
                     "recall_top10": round(fl[yt == 1].mean(), 3)})
fairness = pd.DataFrame(rows).set_index(["attribute", "group"])
fairness

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
lab = [f"{a}: {g}" for a, g in fairness.index]
x = np.arange(len(lab))

axes[0].bar(x - 0.2, fairness["actual_rate"] * 100, 0.4, label="Actual", color="#c1666b")
axes[0].bar(x + 0.2, fairness["predicted_rate"] * 100, 0.4, label="Predicted", color="#2a6f97")
axes[0].set_xticks(x, lab, rotation=45, ha="right")
axes[0].set_ylabel("% readmitted <30 days")
axes[0].set_title("Calibration by group")
axes[0].legend()

axes[1].bar(x, fairness["roc_auc"], color="#2a6f97")
axes[1].axhline(roc_auc_score(y_test, p_test), ls="--", color="grey", label="Overall")
axes[1].set_xticks(x, lab, rotation=45, ha="right")
axes[1].set_ylim(0.5, 0.8)
axes[1].set_title("ROC-AUC by group")
axes[1].legend()
plt.tight_layout()
plt.savefig(FIG / "05_fairness.png", dpi=150)
plt.show()

In [ ]:
fairness.reset_index().to_csv(RES / "05_fairness_by_group.csv", index=False)
importance.head(15).round(4).rename("mean_abs_shap").to_csv(RES / "05_shap_importance.csv")
print("Saved fairness table and SHAP importance")

## 5. Summary (fill in after running)
- **Top drivers of risk:**
- **How prior inpatient visits shape risk:**
- **Fairness: ranking (ROC-AUC) across groups:**
- **Fairness: calibration across groups:**
- **Who lands on the top-10% list:**